<a href="https://colab.research.google.com/github/aishwaryaaramanathan/DL_Lab_241501011/blob/main/dl_exp_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

In [ ]:
# Device and preprocessing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

# Load CIFAR-10 (reuse downloaded dataset)
dataset = datasets.CIFAR10(
    './data', train=True, download=True, transform=transform
)
dataset = torch.utils.data.Subset(dataset, range(500))

train_size = 400
train_data, val_data = random_split(
    dataset, [train_size, 100],
    generator=torch.Generator().manual_seed(42)
)

trainloader = DataLoader(train_data, batch_size=16, shuffle=True)
valloader = DataLoader(val_data, batch_size=16)


In [ ]:
# Load pretrained models
def get_model(name):
    if name == 'vgg':
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(4096, 10)
    elif name == 'resnet':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 10)
    else:
        model = models.googlenet(
            weights=models.GoogLeNet_Weights.DEFAULT,
            aux_logits=True
        )
        model.fc = nn.Linear(model.fc.in_features, 10)
    return model

In [ ]:
# Train and evaluate
def train_model(model, name, epochs=2):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    criterion = nn.CrossEntropyLoss()
    train_acc, val_acc = [], []

    for epoch in range(epochs):
        model.train()
        correct = total = 0

        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            if isinstance(output, tuple):
                output = output[0]
            elif hasattr(output, 'logits'):
                output = output.logits
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            correct += (output.argmax(1) == y).sum().item()
            total += y.size(0)

        train_acc.append(100 * correct / total)

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in valloader:
                x, y = x.to(device), y.to(device)
                output = model(x)
                if isinstance(output, tuple):
                    output = output[0]
                elif hasattr(output, 'logits'):
                    output = output.logits
                correct += (output.argmax(1) == y).sum().item()
                total += y.size(0)

        val_acc.append(100 * correct / total)
        print(f"{name} Epoch {epoch+1}: "
              f"Train={train_acc[-1]:.2f}%, "
              f"Validation={val_acc[-1]:.2f}%")

    return model, train_acc, val_acc

In [ ]:
# Compare models
results = {}
trained_models = {}

for name in ['vgg', 'resnet', 'googlenet']:
    model = get_model(name)
    trained_models[name], train_acc, val_acc = train_model(model, name)
    results[name] = val_acc

In [ ]:
# Plot validation accuracy
for name, accuracy in results.items():
    plt.plot(accuracy, marker='o', label=name.upper())

plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy (%)')
plt.title('VGG vs ResNet vs GoogLeNet')
plt.legend()
plt.grid(True)
plt.show()